In [1]:
import json
import pandas as pd
from pathlib import Path

INPUT_PATH  = Path("outputs/hotels_scraped.json")
OUTPUT_PATH = Path("outputs/hotels_processed.csv")

with open(INPUT_PATH, encoding="utf-8") as f:
    raw = json.load(f)

print(f"Loaded {len(raw)} entries")

Loaded 503 entries


In [2]:
KEEP = ["hotel_id", "hotel_name", "rating", "no_reviews", "hotel_link"]

rows = []
for entry in raw.values():
    meta = entry.get("metadata", {})
    rows.append({k: meta.get(k) for k in KEEP})

df = pd.DataFrame(rows)
print(f"Rows before dedup: {len(df)}")
df.head()

Rows before dedup: 503


,hotel_id,hotel_name,rating,no_reviews,hotel_link
0,3274,Garden Plaza Saigon (Formerly PARKROYAL Saigon),4.3,1726.0,https://www.google.com/maps/place/Garden+Plaza...
1,9698,TTC Hotel Premium - Dalat,4.1,1285.0,https://www.google.com/maps/place/TTC+Hotel+Pr...
2,902,Lotte Hotel Saigon,4.6,6162.0,https://www.google.com/maps/place/Lotte+Hotel+...
3,163,Ramana Hotel Saigon,3.9,2385.0,https://www.google.com/maps/place/Ramana+Hotel...
4,9195,Renaissance Riverside Hotel Saigon,4.4,3942.0,https://www.google.com/maps/place/Renaissance+...


In [3]:
df = df.drop_duplicates(subset="hotel_id", keep="first")
print(f"Rows after dedup:  {len(df)}")
df.head()

Rows after dedup:  503


,hotel_id,hotel_name,rating,no_reviews,hotel_link
0,3274,Garden Plaza Saigon (Formerly PARKROYAL Saigon),4.3,1726.0,https://www.google.com/maps/place/Garden+Plaza...
1,9698,TTC Hotel Premium - Dalat,4.1,1285.0,https://www.google.com/maps/place/TTC+Hotel+Pr...
2,902,Lotte Hotel Saigon,4.6,6162.0,https://www.google.com/maps/place/Lotte+Hotel+...
3,163,Ramana Hotel Saigon,3.9,2385.0,https://www.google.com/maps/place/Ramana+Hotel...
4,9195,Renaissance Riverside Hotel Saigon,4.4,3942.0,https://www.google.com/maps/place/Renaissance+...


In [4]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"Saved {len(df)} rows → {OUTPUT_PATH}")

Saved 503 rows → outputs\hotels_processed.csv


In [5]:
# --- Distance to coastline for the 503 hotels ---
DIST_PATH = Path("../data/hotel_with_distance.csv")

dist_df = pd.read_csv(DIST_PATH, encoding="utf-8-sig", usecols=["hotel_id", "distance2coastline"])

merged = df.merge(dist_df, on="hotel_id", how="left")

missing = merged["distance2coastline"].isna().sum()
print(f"Matched: {len(merged) - missing} / {len(merged)}  (no distance data: {missing})")

merged["distance2coastline"].describe().rename("distance2coastline (m)")

Matched: 503 / 503  (no distance data: 0)


count    503.000000
mean      47.807857
std       54.955843
min        0.000000
25%        0.854500
50%       44.732000
75%       73.252000
max      369.325000
Name: distance2coastline (m), dtype: float64

In [8]:
# Distribution by distance buckets
bins   = [0, 25, 50, 75, 100, 400, 500, float("inf")]
labels = ["<0.1km", "0.1–0.2km", "0.2–0.3km", "0.3–0.4km", "0.4–0.5km", "0.5–1km", ">1km"]

merged["coast_bucket"] = pd.cut(merged["distance2coastline"], bins=bins, labels=labels)
print(merged["coast_bucket"].value_counts().sort_index().to_string())

coast_bucket
<0.1km       184
0.1–0.2km    166
0.2–0.3km     29
0.3–0.4km     93
0.4–0.5km     29
0.5–1km        0
>1km           0
